###Restaurant Reviews

In [118]:
from google.colab import files
uploaded = files.upload()

Saving Restaurant_Reviews.tsv to Restaurant_Reviews (2).tsv


It the restaurant reviews dataset, we have reviews from clients on the experience in the restaurant. There are 1000 reviews and it has only one column that says if it was positive (1) or negative (0) review ('Liked'), and also the review column.

In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.linear_model import LogisticRegression

def remove_punctuation(text):
    import string
    if isinstance(text, str):
      translator = str.maketrans('', '', string.punctuation)
      return text.translate(translator)
    else:
        return ""

restaurant_df = pd.read_csv('Restaurant_Reviews.tsv', sep='\t')
restaurant_df

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone.,0
997,Overall I was not impressed and would not go b...,0
998,"The whole experience was underwhelming, and I ...",0


Some basic information on the data set, we can see that the standard deviation has almost similar number of positive and negative reviews.

In [120]:
restaurant_df.describe()

,Liked
count,1000.00000
mean,0.50000
std,0.50025
min,0.00000
25%,0.00000
50%,0.50000
75%,1.00000
max,1.00000


###Data Preperation

In [121]:
#Removing puntuaction from review column

restaurant_df['Review'] = restaurant_df['Review'].apply(remove_punctuation)
restaurant_df

,Review,Liked
0,Wow Loved this place,1
1,Crust is not good,0
2,Not tasty and the texture was just nasty,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone,0
997,Overall I was not impressed and would not go back,0
998,The whole experience was underwhelming and I t...,0


In [122]:
#Replace all missing (nan) reviews with empty "" string
print(restaurant_df[restaurant_df['Review'].isna()])
#There is no nan reviews, otherwise we would replace them using fillna.("")

Empty DataFrame
Columns: [Review, Liked]
Index: []


In [123]:
#Positive 1, negative -1

restaurant_df['Liked'] = np.where(restaurant_df['Liked'] == 0, -1, 1)
restaurant_df

,Review,Liked
0,Wow Loved this place,1
1,Crust is not good,-1
2,Not tasty and the texture was just nasty,-1
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,-1
996,Appetite instantly gone,-1
997,Overall I was not impressed and would not go back,-1
998,The whole experience was underwhelming and I t...,-1


###Dividing data

In [124]:
from sklearn.model_selection import train_test_split

X = restaurant_df['Review']
y = restaurant_df['Liked']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#80% goes to X_train to train data
#20% stays around to test if this model is accurate (X_test)

print("Train size: ", len(X_train), len(y_train))
print("Test size: ", len(X_test), len(y_test))

Train size:  800 800
Test size:  200 200


###CountVectorizer

CountVectorizer changes text into numbers. Each review becomes vector of numbers. It creates a dictionary of unique words.

In [125]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Number of unique words", len(vectorizer.get_feature_names_out()),'\n')
print("Vectorizer creates a matrix of unique words where all of them are columns \n")
print("Some of the words:", vectorizer.get_feature_names_out()[25:35])



Number of unique words 1813 

Vectorizer creates a matrix of unique words where all of them are columns 

Some of the words: ['accordingly' 'accountant' 'acknowledged' 'actual' 'actually' 'added'
 'affordable' 'after' 'afternoon' 'again']


###Logistic Regression

Logistic regresion sums all the weights that the words have. It is a little bit as if every word would vote for positive or negative review and model sums all the votes and decides which has what weight.

In [126]:
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)

coefficients = model.coef_[0]
print("Some coefficients: ", coefficients, " \n")


unique_words = vectorizer.get_feature_names_out()


unique_words_weights = list(zip(unique_words, coefficients))
print("List of a unique words zipped with weights: ", unique_words_weights[:10], " \n")


sorted_unique_words_weights = sorted(unique_words_weights, key=lambda x: x[1], reverse=True)



top_positive = sorted_unique_words_weights[:10]

top_negative = sorted_unique_words_weights[-10:]

print("Top positive words: ", top_positive)
print("Top negative words: ", top_negative)

Some coefficients:  [-0.26895668 -0.04764625 -0.05489591 ... -0.2823967   0.43331661
 -0.67997553]  

List of a unique words zipped with weights:  [('10', np.float64(-0.26895668004715517)), ('100', np.float64(-0.047646247040587515)), ('12', np.float64(-0.05489590642882068)), ('15lb', np.float64(-0.12345860468144558)), ('17', np.float64(-0.07492698840295654)), ('1979', np.float64(-0.2607047985806425)), ('20', np.float64(-0.14995416846103918)), ('2007', np.float64(0.07472162564283755)), ('30', np.float64(-0.3486251965253449)), ('30s', np.float64(-0.13575865159705924))]  

Top positive words:  [('great', np.float64(2.5885341901617194)), ('good', np.float64(1.8871355680034558)), ('delicious', np.float64(1.8685608070307307)), ('awesome', np.float64(1.5001058608152467)), ('amazing', np.float64(1.470161005759919)), ('nice', np.float64(1.2954929412538716)), ('love', np.float64(1.2619621844116566)), ('friendly', np.float64(1.1690644148019966)), ('fantastic', np.float64(1.0809866174730172)), ('y

###Prediting sentiment

In [127]:
# Predict the sentiment of test data reviews.

y_pred = model.predict(X_test_vectorized)
print("Predykcje klas dla testowych recenzji:", y_pred[:10], "\n")

#Predict the sentiment of test data reviews in terms of probability.
y_prob = model.predict_proba(X_test_vectorized)
print("Predykcje prawdopodobieństw dla pierwszych 10 recenzji:\n", y_prob[:10])



Predykcje klas dla testowych recenzji: [ 1  1  1  1  1  1 -1  1 -1  1] 

Predykcje prawdopodobieństw dla pierwszych 10 recenzji:
 [[0.40874783 0.59125217]
 [0.09090415 0.90909585]
 [0.08103493 0.91896507]
 [0.03130727 0.96869273]
 [0.34906034 0.65093966]
 [0.03910975 0.96089025]
 [0.73468185 0.26531815]
 [0.40006389 0.59993611]
 [0.57719033 0.42280967]
 [0.28039682 0.71960318]]


In [128]:
#Find five most positive and most negative reviews.

test_df = pd.DataFrame({
    'Review': X_test,
    'TrueLabel': y_test,
    'PosProbability': y_prob[:, 1]
})

most_positive_reviews = test_df.sort_values('PosProbability', ascending=False).head(5)
most_negative_reviews = test_df.sort_values('PosProbability', ascending=True).head(5)

print("5 najbardziej pozytywnych recenzji:\n", most_positive_reviews[['Review','PosProbability']])
print("\n5 najbardziej negatywnych recenzji:\n", most_negative_reviews[['Review','PosProbability']])

5 najbardziej pozytywnych recenzji:
                                                 Review  PosProbability
292  The staff is great the food is delish and they...        0.996855
55   Loved itfriendly servers great food wonderful ...        0.987809
879  Now the burgers arent as good the pizza which ...        0.972942
660  I personally love the hummus pita baklava fala...        0.968693
174  Everything on the menu is terrific and we were...        0.968187

5 najbardziej negatywnych recenzji:
                                                 Review  PosProbability
584  After I pulled up my car I waited for another ...        0.010833
275  Ive had better not only from dedicated boba te...        0.013201
849  Bad day or not I have a very low tolerance for...        0.018809
261  I have been in more than a few bars in Vegas a...        0.022325
280  I went to Bachi Burger on a friends recommenda...        0.023414


In [129]:
#Calculate the accuracy of predictions.

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Dokładność predykcji: {:.2f}%".format(accuracy * 100))

Dokładność predykcji: 81.00%


###Limited Dictionary

In [130]:
from sklearn.feature_extraction.text import CountVectorizer

# Limiting dictionary to 500 most used words
limited_vectorizer = CountVectorizer(max_features=500)

X_train_limited = limited_vectorizer.fit_transform(X_train)
X_test_limited = limited_vectorizer.transform(X_test)

# Checking the dict
print("Liczba słów w ograniczonym słowniku:", len(limited_vectorizer.get_feature_names_out()))
print("Pierwsze 10 słów:", limited_vectorizer.get_feature_names_out()[:10])

Liczba słów w ograniczonym słowniku: 500
Pierwsze 10 słów: ['10' '20' '30' '35' '40' 'about' 'absolutely' 'added' 'after' 'again']


Training the logistic regression model on the training data. Predicting the sentiment for test reviews. And predicting the probability. Of course i have a dataframe also wihich helps to analyze the output.

In [131]:
from sklearn.linear_model import LogisticRegression

model_limited = LogisticRegression()
model_limited.fit(X_train_limited, y_train)

y_pred_limited = model_limited.predict(X_test_limited)
y_prob_limited = model_limited.predict_proba(X_test_limited)


test_df_limited = pd.DataFrame({
    'Review': X_test,
    'TrueLabel': y_test,
    'PosProbability': y_prob_limited[:, 1]
})


most_positive_reviews = test_df_limited.sort_values('PosProbability', ascending=False).head(5)
most_negative_reviews = test_df_limited.sort_values('PosProbability', ascending=True).head(5)

print("\n5 najbardziej pozytywnych recenzji:\n", most_positive_reviews[['Review','PosProbability']])
print("\n5 najbardziej negatywnych recenzji:\n", most_negative_reviews[['Review','PosProbability']])


accuracy_limited = accuracy_score(y_test, y_pred_limited)
print("\nDokładność modelu z ograniczonym słownikiem: {:.2f}%".format(accuracy_limited * 100))


coefficients_limited = model_limited.coef_[0]
tokens_limited = limited_vectorizer.get_feature_names_out()
token_weights_limited = list(zip(tokens_limited, coefficients_limited))
sorted_tokens_limited = sorted(token_weights_limited, key=lambda x: x[1], reverse=True)

top_positive_words = sorted_tokens_limited[:10]
top_negative_words = sorted_tokens_limited[-10:]

print("\nTop 10 pozytywnych słów w ograniczonym słowniku:\n", top_positive_words)
print("\nTop 10 negatywnych słów w ograniczonym słowniku:\n", top_negative_words)


5 najbardziej pozytywnych recenzji:
                                                 Review  PosProbability
292  The staff is great the food is delish and they...        0.997601
55   Loved itfriendly servers great food wonderful ...        0.992988
174  Everything on the menu is terrific and we were...        0.975373
879  Now the burgers arent as good the pizza which ...        0.973675
689                           Good food  good service         0.965929

5 najbardziej negatywnych recenzji:
                                                 Review  PosProbability
584  After I pulled up my car I waited for another ...        0.008099
261  I have been in more than a few bars in Vegas a...        0.011277
275  Ive had better not only from dedicated boba te...        0.012209
280  I went to Bachi Burger on a friends recommenda...        0.014954
849  Bad day or not I have a very low tolerance for...        0.021657

Dokładność modelu z ograniczonym słownikiem: 79.00%

Top 10 pozytywnych

##Hashing Vectorizer

Creating HashingVectorizer, which changes every review into vector of numbers like the previous one.

Key difference is that it does not create a dictionary, it uses hashing function to put every word in some column.

In [132]:
from sklearn.feature_extraction.text import HashingVectorizer

#create the vectorizer
hash_vectorizer = HashingVectorizer()

#transforming data
X_train_hash = hash_vectorizer.transform(X_train)
X_test_hash = hash_vectorizer.transform(X_test)

#training data
from sklearn.linear_model import LogisticRegression
model_hash = LogisticRegression()
model_hash.fit(X_train_hash, y_train)

#predict
y_pred_hash = model_hash.predict(X_test_hash)
y_prob_hash = model_hash.predict_proba(X_test_hash)

#real dataframe to analyze
test_df_hash = pd.DataFrame({
    'Review': X_test,
    'TrueLabel': y_test,
    'PredictedLabel': y_pred_hash,
    'PosProbability': y_prob_hash[:, 1]
})

# Wyświetlamy pierwsze 10 wierszy
test_df_hash.head(10)

,Review,TrueLabel,PredictedLabel,PosProbability
521,If you havent gone here GO NOW,1,1,0.532254
737,Try them in the airport to experience some tas...,1,1,0.660927
740,The restaurant is very clean and has a family ...,1,1,0.674339
660,I personally love the hummus pita baklava fala...,1,1,0.731395
411,Come hungry leave happy and stuffed,1,1,0.578631
678,Its a great place and I highly recommend it,1,1,0.814811
626,Best of luck to the rude and noncustomer servi...,-1,-1,0.447564
513,Reasonably priced also,1,1,0.544826
859,Worst foodservice Ive had in a while,-1,1,0.502031
136,I had a seriously solid breakfast here,1,1,0.610185


The model look on vector of numbers from review. Each column has its own weight, which model learns while training. Using weighted sum the model calculates and changes it to probabbility.

In [133]:
most_positive = test_df_hash.sort_values('PosProbability', ascending=False).head(5)
most_negative = test_df_hash.sort_values('PosProbability', ascending=True).head(5)

print("5 najbardziej pozytywnych recenzji:\n", most_positive[['Review','PosProbability']])
print("\n5 najbardziej negatywnych recenzji:\n", most_negative[['Review','PosProbability']])

5 najbardziej pozytywnych recenzji:
                                                 Review  PosProbability
158                                 this place is good        0.880756
292  The staff is great the food is delish and they...        0.872894
899                         Overall a great experience        0.871590
689                           Good food  good service         0.867075
55   Loved itfriendly servers great food wonderful ...        0.836991

5 najbardziej negatywnych recenzji:
                                                 Review  PosProbability
820          But I definitely would not eat here again        0.125593
439                 If youre not familiar check it out        0.140062
613  Sorry I will not be getting food from here any...        0.149078
679                 Service was slow and not attentive        0.155623
941                 Probably not in a hurry to go back        0.166540


In [134]:
from sklearn.metrics import accuracy_score

accuracy_hash = accuracy_score(y_test, y_pred_hash)
print("Dokładność modelu z HashingVectorizer: {:.2f}%".format(accuracy_hash * 100))

Dokładność modelu z HashingVectorizer: 77.00%


Why the accuracy is lower than i CountVectorizer? Because in hashing every word has its own column. The model specifaclly knows which word impacts on outcome.

In hashin vectorizer words are hashed - different words can be in the same column.